### Step 1: Set up the environment
We start by preparing the Colab environment for training.


### Step 2: Upload datasets
We upload the fine-grained hope speech datasets.


### Step 3: Install libraries
We install the libraries needed for model training.


In [ ]:
!pip install transformers datasets accelerate scikit-learn


### Step 4: Import libraries
We import all necessary Python packages.


In [ ]:
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer


### Step 5: Load datasets
We load the training, development, and test data.


In [ ]:
train_df = pd.read_csv("Finegrained_train_data.csv")
dev_df = pd.read_csv("Finegrained_dev_data.csv")
test_df = pd.read_csv("Finegrained_test_data_withoutlabel.csv")

print(train_df.head())


### Step 6: Initialize model
We use a multilingual transformer suitable for code-mixed text.


In [ ]:
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

labels = train_df["Label"].unique().tolist()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)

print(label2id)


### Step 7: Tokenize text
We convert text into numbers the model can understand.


In [ ]:
def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_data(train_df["Text"])
dev_encodings = tokenize_data(dev_df["Text"])


### Step 8: Prepare labels
We convert label names into numeric form.


In [ ]:
train_labels = torch.tensor([label2id[l] for l in train_df["Label"]])
dev_labels = torch.tensor([label2id[l] for l in dev_df["Label"]])


### Step 9: Create dataset class
We wrap the data so the trainer can use it.


In [ ]:
class HopeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = HopeDataset(train_encodings, train_labels)
dev_dataset = HopeDataset(dev_encodings, dev_labels)


### Step 10: Train the model
We fine-tune the model on the training data.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="no",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

trainer.train()


### Step 11: Prepare test data
We convert the test comments into a format the model understands.


In [ ]:
test_encodings = tokenizer(
    test_df["Text"].tolist(),
    padding=True,
    truncation=True,
    max_length=128
)


### Step 12: Create test dataset
We wrap the test data so the trained model can read it.


In [ ]:
test_dataset = HopeDataset(test_encodings)


### Step 13: Predict labels
The model predicts the fine-grained hope type for each comment.


In [ ]:
predictions = trainer.predict(test_dataset)
pred_ids = np.argmax(predictions.predictions, axis=1)
pred_labels = [id2label[i] for i in pred_ids]


### Step 14: Create submission file
We prepare the final CSV file in the required format.


In [ ]:
submission = pd.DataFrame({
    "id": range(1, len(pred_labels) + 1),
    "label": pred_labels
})

submission.head()


### Step 15: Save CSV file
This file will be submitted for Track 2 evaluation.


In [ ]:
submission.to_csv("PrimeLine_Tulu_task2.csv", index=False)
